### Assignment 4: Text Classification with Naive Bayes

This assignment implements a Naive Bayes classifier from scratch to classify customer reviews as positive or negative.

The dataset contains 11,914 reviews: 6,000 positive and 5,914 negative. The review text is already lowercased and tokenized. Each line contains a topic, sentiment label, document identifier, and review text. Only the sentiment label and review text are used for sentiment classification.

In [18]:
import numpy as np

from pathlib import Path 
from collections import Counter

In [19]:
# load data through the path 
DATA_PATH = Path("A4_data") / "all_sentiment_shuffled.txt"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find the dataset at: {DATA_PATH.resolve()}\n"
        "Make sure this notebook is saved and running from the A4 folder."
    )
print(f"Dataset found at: {DATA_PATH.resolve()}")

Dataset found at: /Users/heeyashah/UWaterloo/4th year/syde572/syde572-repo/A4/A4_data/all_sentiment_shuffled.txt


In [20]:
def read_documents(file_path):
    """
    Read the data, seperate into list of words with labels indicating pos / neg
    """
    documents = []
    labels = []

    # open the file
    with Path(file_path).open(mode="r", encoding="utf-8") as file:
        # go through each line
        for line_number, line in enumerate(file, start=1):
            # clean up line & divide into seperate pieces
            parts = line.strip().split()
            # skip if empty
            if not parts:
                continue
            # check to make sure all lines are valid
            if len(parts) < 4:
                raise ValueError(
                    f"Line {line_number} is malformed: {line!r}"
                )

            # get the sentiment label (col 1)
            sentiment_label = parts[1]
            # get the words (starts from col 3)
            review_words = parts[3:]
            # confirm label matches expected values
            if sentiment_label not in {"pos", "neg"}:
                raise ValueError(
                    f"Unexpected label {sentiment_label!r} "
                    f"on line {line_number}"
                )
            labels.append(sentiment_label)
            documents.append(review_words)
    return documents, labels

In [21]:
# Call the function to read every document and label
all_docs, all_labels = read_documents(DATA_PATH)
assert len(all_docs) == len(all_labels)

print(f"Number of documents: {len(all_docs)}")
print(f"Label counts: {Counter(all_labels)}")

Number of documents: 11914
Label counts: Counter({'pos': 6000, 'neg': 5914})


The dataset is shuffled already so it is split into 9,531 training reviews and 2,383 evaluation reviews.

In [22]:
# create 80/20 split of data
split_point = int(0.80 * len(all_docs))
# first 80% is training 
train_docs = all_docs[:split_point]
train_labels = all_labels[:split_point]
# remaining 20% is evaluation
eval_docs = all_docs[split_point:]
eval_labels = all_labels[split_point:]

The dataset has now been divided into four variables:

- `train_docs`: used to train the Naive Bayes classifier
- `train_labels`: correct labels for the training data
- `eval_docs`: unseen data used to evaluate the classifier
- `eval_labels`: correct labels for the evaluation data

Only the training data will be used to estimate the Naive Bayes probabilities. The evaluation data must not be used during training because that would cause data leakage and produce an overly optimistic accuracy result.

### Training the Naive Bayes model

Each review is represented by its word counts, with word order ignored. Naive Bayes assumes that words are conditionally independent given the sentiment class.

For each sentiment class, the model needs to calculate:

1. The number of training documents in the class
2. The number of times each word occurs in the class
3. The total number of word occurrences in the class
4. The prior probability of the class
5. The conditional probability of each word given the class

The prior probability of class $c$ is:

$$
P(c) =
\frac{\text{number of training documents in class } c}
{\text{total number of training documents}}
$$

The probability of a word given a class is:

$$
P(w \mid c) =
\frac{\operatorname{count}(w,c) + \alpha}
{\operatorname{totalWords}(c) + \alpha |V|}
$$

where:

- \(c\) is a sentiment class
- \(w\) is a word
- \(V\) is the training vocabulary
- $\alpha$ is the smoothing parameter

Laplace smoothing is applied during training with $\alpha$ = 1. This adds one to each word’s count in each class and adds |V| to the denominator to keep the probabilities normalized. It prevents a word from receiving zero probability when it appears in the training vocabulary but has no occurrences in a particular class.

In [23]:
def train_nb(documents, labels, alpha=1.0):
    """
    Train a multinomial Naive Bayes text classifier
    """

   # validate inputs 
    if len(documents) != len(labels):
        raise ValueError(
            "The number of documents must equal the number of labels."
        )
    if len(documents) == 0:
        raise ValueError("The training set cannot be empty.")
    if alpha <= 0:
        raise ValueError("The smoothing parameter alpha must be positive.")

    # get the class label
    classes = sorted(set(labels))
    # count number of documents per label
    document_counts = Counter(labels)
    # count number of pos / neg words
    word_counts = {
        label: Counter()
        for label in classes
    }
    # create a vocabulary list for all unique words 
    vocabulary = set()

    for document, label in zip(documents, labels):
        # Add the document's words to the vocabulary
        vocabulary.update(document)
        # update word counter per label
        word_counts[label].update(document)
    vocabulary_size = len(vocabulary)

    # count the total number of word occurences per class
    total_word_counts = {
        label: sum(word_counts[label].values())
        for label in classes
    }
    # calculate logarithm per class 
    number_of_documents = len(documents)
    log_priors = {
        label: np.log(
            document_counts[label] / number_of_documents
        )
        for label in classes
    }
    # calculate logarithm of P(word | class)
    log_word_probabilities = {
        label: {}
        for label in classes
    }

    for label in classes:
        # denominator is the total number of words per class & smoothing term
        denominator = (
            total_word_counts[label]
            + alpha * vocabulary_size
        )
        # go through every word in the vocabulary
        for word in vocabulary:
            word_count = word_counts[label][word]
            # Laplace-smoothed estimate of P(word | class)
            probability = (word_count + alpha) / denominator
            # store log probabilities
            log_word_probabilities[label][word] = np.log(probability)
    # Store all learned information together in one dictionary
    model = {
        "classes": classes,
        "document_counts": document_counts,
        "word_counts": word_counts,
        "total_word_counts": total_word_counts,
        "vocabulary": vocabulary,
        "vocabulary_size": vocabulary_size,
        "log_priors": log_priors,
        "log_word_probabilities": log_word_probabilities,
        "alpha": alpha
    }
    return model

In [24]:
# train the classifier using training data
nb_model = train_nb(
    documents=train_docs,
    labels=train_labels,
    alpha=1.0
)

## Inspecting learned word counts

To make sure the model training was successful, the counts of words that are expected to have positive or negative meanings will now be compared.

For example, `great` should occur more frequently in positive reviews, while `bad` should generally occur more frequently in negative reviews.

These counts alone do not make the final prediction. The classifier will use the smoothed probabilities derived from these counts.

In [25]:
words_to_check = ["great", "bad", "excellent", "terrible"]

for word in words_to_check:
    print(f"Word: {word!r}")
    for label in nb_model["classes"]:
        count = nb_model["word_counts"][label][word]
        log_probability = nb_model["log_word_probabilities"][label][word]
        # Convert the log probability back to an ordinary probability
        probability = np.exp(log_probability)
        print(
            f"  {label}: "
            f"count = {count}, "
            f"P(word | class) = {probability:.8f}"
        )
    print()

Word: 'great'
  neg: count = 824, P(word | class) = 0.00107519
  pos: count = 2008, P(word | class) = 0.00259800

Word: 'bad'
  neg: count = 709, P(word | class) = 0.00092531
  pos: count = 261, P(word | class) = 0.00033881

Word: 'excellent'
  neg: count = 99, P(word | class) = 0.00013033
  pos: count = 441, P(word | class) = 0.00057159

Word: 'terrible'
  neg: count = 171, P(word | class) = 0.00022416
  pos: count = 32, P(word | class) = 0.00004267



## Scoring a document

The trained model can now calculate how well a document matches a particular sentiment label.

For a document \(d\) and class \(c\), the Naive Bayes score is:

$$
\operatorname{score}(d,c)
=
\log P(c)
+
\sum_{w\in d}\log P(w\mid c)
$$

The calculation begins with the log prior of the class. It then adds the log probability of every word in the document. Logarithmic probabilities are used to avoid problems with numeric precision when multiplying many small probabilities. Using logarithms replaces multiplication with addition.

$$
\log\left(P(c)\prod_{w\in d}P(w\mid c)\right)
=
\log P(c)+\sum_{w\in d}\log P(w\mid c)
$$

If a word appears multiple times, its log probability is added multiple times. Therefore, repeated words have a greater influence on the score.

In [26]:
def score_doc_label(document, label, model):
    """
    Calculate the Naive Bayes log score of a document for one label.
    """

    # get the log prior of the class 
    score = model["log_priors"][label]

    # Add the log probability of each word given the proposed class
    for word in document:
        if word in model["vocabulary"]:
            score += model["log_word_probabilities"][label][word]
    return score

### Handling words outside the vocabulary

A new review may contain a word that never appeared in the training data.

This implementation ignores out-of-vocabulary words because the model has no training evidence about whether they are positive or negative.

Laplace smoothing still handles words that are in the overall vocabulary but did not occur in one particular class. For example, if a word appeared in positive training documents but never in negative training documents, smoothing gives it a small nonzero negative probability.

### Sanity check 1: Expected scores for `great` and `bad`

The word `great` should receive a higher positive score than negative score. Conversely, the word `bad` should receive a higher negative score than positive score.

The log scores can be converted back into probabilities using `np.exp()`. The probability of a positive document containing only `great` is expected to be approximately 0.001–0.002, depending on the smoothing parameter.

In [27]:
document = ["great"]

great_pos_score = score_doc_label(
    document,
    "pos",
    nb_model
)

great_neg_score = score_doc_label(
    document,
    "neg",
    nb_model
)

print(f"Positive log score: {great_pos_score:.6f}")
print(f"Negative log score: {great_neg_score:.6f}")

if great_pos_score > great_neg_score:
    print("The model considers 'great' more positive.")
else:
    print("The model considers 'great' more negative.")

Positive log score: -6.629204
Negative log score: -7.545657
The model considers 'great' more positive.


In [32]:
document = ["bad"]

bad_pos_score = score_doc_label(
    document,
    "pos",
    nb_model
)

bad_neg_score = score_doc_label(
    document,
    "neg",
    nb_model
)

print(f"Positive log score: {bad_pos_score:.6f}")
print(f"Negative log score: {bad_neg_score:.6f}")

if bad_neg_score > bad_pos_score:
    print("The model considers 'bad' more negative.")
else:
    print("The model considers 'bad' more positive.")

Positive log score: -8.666252
Negative log score: -7.695776
The model considers 'bad' more negative.


In [33]:
assert great_pos_score > great_neg_score
assert bad_neg_score > bad_pos_score

print("Sanity check 1 passed.")

Sanity check 1 passed.


In [45]:
print(f"'great': pos = {np.exp(great_pos_score):.6f}, "
      f"neg = {np.exp(great_neg_score):.6f}")
print(f"'bad': pos = {np.exp(bad_pos_score):.6f}, "
      f"neg = {np.exp(bad_neg_score):.6f}")

'great': pos = 0.001321, neg = 0.000528
'bad': pos = 0.000172, neg = 0.000455


For “great,” the positive joint probability is approximately 0.001321, compared with 0.000528 for negative. This falls within the assignment’s expected range of 0.001–0.002. For “bad,” the negative value is higher than the positive value. Both results match the expected sentiment associations. These values include the class prior; they are not normalized posterior probabilities.

### Sanity check 2: Handling words outside the vocabulary

The `score_doc_label` function should successfully score the document `["a", "top-quality", "performance"]` without crashing, even if one of its words did not appear in the training vocabulary.

In [34]:
test_document = [
    "a",
    "top-quality",
    "performance"
]

for label in nb_model["classes"]:
    score = score_doc_label(
        test_document,
        label,
        nb_model
    )

    print(f"{label} log score: {score:.6f}")

    # Confirm that the function returned a valid finite number.
    assert np.isfinite(score)

print("Sanity check 2 passed.")

neg log score: -13.486892
pos log score: -12.807858
Sanity check 2 passed.


## Classifying a new document

To classify a document, the classifier calculates its score for every possible class.

For two sentiment classes, it compares
$\operatorname{score}(d,\text{pos})$ and $\operatorname{score}(d,\text{neg})$

The predicted label is the class with the larger log score:

$$
\hat{c}
=
\underset{c}{\operatorname{argmax}}
\ \operatorname{score}(d,c)
$$

In [29]:
def classify_nb(document, model):
    """
    classify one document using a trained Naive Bayes model
    """

    # store the score calculated for each class
    scores = {}
    # calculate how well the document matches each class
    for label in model["classes"]:
        scores[label] = score_doc_label(
            document,
            label,
            model
        )
    predicted_label = max(scores, key=scores.get)

    return predicted_label

### Testing `classify_nb` on small documents

The classifier is tested on two small documents containing words with clearly positive or negative sentiment. The expected predictions are `pos` for the positive example and `neg` for the negative example.

In [35]:
positive_document = [
    "great",
    "excellent",
    "recommend"
]

negative_document = [
    "bad",
    "terrible",
    "waste"
]

print(
    "Positive document prediction:",
    classify_nb(positive_document, nb_model)
)

print(
    "Negative document prediction:",
    classify_nb(negative_document, nb_model)
)

Positive document prediction: pos
Negative document prediction: neg


## Classifying the evaluation documents

The `classify_nb` function classifies one document at a time. To evaluate the model, the classifier must be applied to every document in the evaluation set.

The function `classify_documents` loops through the collection, calls `classify_nb` for each document, and stores the predicted labels in a list.

The predictions are stored in the same order as the evaluation documents. Therefore, each predicted label can later be compared with the true label at the same position.

In [31]:
def classify_documents(documents, model):
    """
    Classify every document in a collection.
    """
    predictions = []
    # Classify each document individually
    for document in documents:
        predicted_label = classify_nb(
            document,
            model
        )
        predictions.append(predicted_label)
    return predictions

In [36]:
# classify evaluation data 
eval_predictions = classify_documents(
    eval_docs,
    nb_model
)
# amke sure that all the evaluation documents have predictions
print(f"Number of evaluation documents: {len(eval_docs)}")
print(f"Number of predictions: {len(eval_predictions)}")

Number of evaluation documents: 2383
Number of predictions: 2383


In [ ]:
# print the first 10 predictions to make sure they look correct
print(eval_predictions[:10])
print(eval_labels[:10])

['neg', 'pos', 'neg', 'pos', 'pos', 'neg', 'pos', 'neg', 'neg', 'pos']
['neg', 'pos', 'neg', 'pos', 'pos', 'neg', 'pos', 'neg', 'neg', 'neg']


## Evaluating the classifier

The classifier is evaluated using accuracy, which is the number of correctly classified documents divided by the total number of evaluation documents.

$$
\text{accuracy}
=
\frac{\text{number of correct predictions}}
{\text{total number of predictions}}
$$

In [40]:
def accuracy(true_labels, guessed_labels):
    """
    Calculate the proportion of correctly predicted labels
    """
    correct_predictions = 0
    # Compare each prediction with its corresponding true label
    for true_label, guessed_label in zip(
        true_labels,
        guessed_labels
    ):
        if true_label == guessed_label:
            correct_predictions += 1
    return correct_predictions / len(true_labels)

# calculate evaluation accuracy 
eval_accuracy = accuracy(
    eval_labels,
    eval_predictions
)

print(f"Evaluation accuracy: {eval_accuracy:.4f}")
print(f"Evaluation accuracy: {eval_accuracy:.2%}")

Evaluation accuracy: 0.8154
Evaluation accuracy: 81.54%


The classifier correctly classified 81.54% of the evaluation reviews. This measures performance on reviews held out from training within the supplied dataset.

## Error Analysis 

A few incorrectly classified evaluation documents are examined to understand the limitations of the Naive Bayes classifier.

Short documents where the classifier strongly preferred the incorrect class are selected. The difference between the positive and negative log scores indicates how strongly the classifier preferred one class over the other.

In [43]:
misclassified_indices = []

# Compare each predicted label with its true label
for index, (true_label, predicted_label) in enumerate(
    zip(eval_labels, eval_predictions)):
    if true_label != predicted_label:
        misclassified_indices.append(index)

print(
    "Number of misclassified documents:",
    len(misclassified_indices)
)

short_errors = []

for index in misclassified_indices:
    document = eval_docs[index]

    # Only consider relatively short documents so they are easier
    # to inspect and discuss.
    if len(document) <= 40:
        pos_score = score_doc_label(
            document,
            "pos",
            nb_model
        )

        neg_score = score_doc_label(
            document,
            "neg",
            nb_model
        )

        # A larger score difference means the classifier had a
        # stronger preference between the two classes.
        score_difference = abs(
            pos_score - neg_score
        )

        short_errors.append(
            (
                score_difference,
                index,
                pos_score,
                neg_score
            )
        )

# Sort from largest score difference to smallest.
short_errors.sort(reverse=True)

Number of misclassified documents: 440


In [44]:
selected_errors = short_errors[:3]

for example_number, error in enumerate(
    selected_errors,
    start=1
):
    score_difference, index, pos_score, neg_score = error

    print(f"EXAMPLE {example_number}")
    print(f"True label: {eval_labels[index]}")
    print(f"Predicted label: {eval_predictions[index]}")
    print(f"Positive score: {pos_score:.3f}")
    print(f"Negative score: {neg_score:.3f}")
    print(f"Score difference: {score_difference:.3f}")
    print(f"Number of words: {len(eval_docs[index])}")

    print("Review:")
    print(" ".join(eval_docs[index]))

    print("-" * 80)

EXAMPLE 1
True label: pos
Predicted label: neg
Positive score: -255.966
Negative score: -247.879
Score difference: 8.087
Number of words: 37
Review:
i bought zonealarm internet security suite for the included zonealarm pro firewall ( it cost much less that way ) . if i did n't already have norton antivirus , i 'd use the antivirus , too
--------------------------------------------------------------------------------
EXAMPLE 2
True label: pos
Predicted label: neg
Positive score: -185.483
Negative score: -180.003
Score difference: 5.480
Number of words: 35
Review:
this movie is a reallllllllllllly scary movie . the only part i did n't like was that pervert mikks in the beginning . be warned of grossness . that is all i need to say
--------------------------------------------------------------------------------
EXAMPLE 3
True label: pos
Predicted label: neg
Positive score: -240.699
Negative score: -235.966
Score difference: 4.733
Number of words: 40
Review:
excellent service , and the pro

### Discussion of misclassified documents

#### Example 1

The true label was `pos`, but the classifier predicted `neg`. The positive opinion in this review is mostly implied rather than stated using many clearly positive words. Words related to cost, security, and antivirus may have been more strongly associated with negative reviews in the training data.

#### Example 2

The true label was `pos`, but the classifier predicted `neg`. The review describes the movie as scary but appears to use `scary` positively because it is reviewing a horror movie. It also contains negative-sounding expressions such as `didn't like`, `pervert`, `grossness`, and `be warned`. Naive Bayes considers the individual words but does not understand that being scary can be desirable for this type of movie.

#### Example 3

The true label was `pos`, but the classifier predicted `neg`. This is a mixed review. It contains positive phrases such as `excellent service` and `worked extremely well`, but it also contains negative words such as `criticism`, `cost`, and `ridiculous`. In the trained model, `postage` and `ridiculous` contribute to a negative prediction, which outweigh the positive words such as `excellent` and `well`. Since the classifier combines individual words without distinguishing the overall recommendation from complaint, it predicts the wrong overall sentiment.

### Conclusion 

The Naive Bayes classifier achieved 81.54% accuracy on the evaluation set. Laplace smoothing prevented zero word probabilities, while logarithms allowed documents to be scored without multiplying many small probabilities. The misclassified examples illustrate limitations in handling mixed sentiment, negation, and words whose meaning depends on context.